### 1 - Load Data

In [7]:
%pip install faiss-cpu
#!pip install faiss-gpu

Note: you may need to restart the kernel to use updated packages.


In [8]:
import numpy as np
import pandas as pd
import faiss
import time

# load vector (SAMA seperti Annoy)
X = np.load("../0_data/processed/features.npy")

# load metadata (opsional kalau mau tampilkan lagu)
meta = pd.read_csv("../0_data/processed/meta.csv")

# query
query_idx = 10
query = X[query_idx].reshape(1, -1)

# memeriksa sinkronisasi data dan metadata
assert len(X) == len(meta) 

print(X.shape)
meta.head()

(232725, 9)


,track_name,artist_name,genre
0,C'est beau de faire un Show,Henri Salvador,Movie
1,Perdu d'avance (par Gad Elmaleh),Martin & les fées,Movie
2,Don't Let Me Be Lonely Tonight,Joseph Williams,Movie
3,Dis-moi Monsieur Gordon Cooper,Henri Salvador,Movie
4,Ouverture,Fabien Nataf,Movie


### 2 - Build FAISS

In [9]:
def build_faiss(X):
    dim = X.shape[1]
    index = faiss.IndexFlatL2(dim)

    start = time.time()
    index.add(X)
    build_time = time.time() - start

    return index, build_time

### 3 - Query FAISS

In [10]:
def query_faiss(index, query, k=20):
    start = time.time()
    distances, indices = index.search(query, k)
    query_time = time.time() - start

    return indices, distances, query_time

### 4 - Hasil Rekomendasi

In [12]:
index, build_time = build_faiss(X)

indices, distances, query_time = query_faiss(index, query, k=20)

print("=== FAISS ===")
print(f"Build time: {build_time:.4f}")
print(f"Query time: {query_time:.6f}")

print("\n🎵 Query:")
print(meta.iloc[query_idx]["track_name"], "-", meta.iloc[query_idx]["artist_name"], "-", meta.iloc[query_idx]["genre"])

print("\n🎧 Recommendations:")
for i in indices[0]:
    if i == query_idx:
        continue
    print("-", meta.iloc[i]["track_name"], "-", meta.iloc[i]["artist_name"], "-", meta.iloc[query_idx]["genre"])

=== FAISS ===
Build time: 0.0180
Query time: 0.003134

🎵 Query:
Symphony No.4 In E Minor Op.98 : IV. Allegro Energico E Passionato - Leopold Stokowski - Movie

🎧 Recommendations:
- My Favorite Day - Geoff Zanelli - Movie
- Sinai Desert - Johan Söderqvist - Movie
- Sheik's Devotion - Rozen - Movie
- La Gioconda, Op.9 / Act 2: "Cielo e mar" - Amilcare Ponchielli - Movie
- Madama Butterfly: Entrance of Butterfly - Giacomo Puccini - Movie
- Messa da Requiem: 2. Confutatis - Giuseppe Verdi - Movie
- Full Apple Tree - Dean Evenson - Movie
- Boston - Justin Hurwitz - Movie
- Giordano : Andrea Chénier : Act 4 "Come un bel dì di maggio" [Andrea Chénier] - Umberto Giordano - Movie
- The Tear Heals - Mandy Moore - Movie
- Delius: Koanga: La Calinda - Frederick Delius - Movie
- Do You Hear That? / I Wonder - Mary Costa - Movie
- Pagliacci, Act 1: Recitar!....Vesti la giubba - Ruggero Leoncavallo - Movie
- Leoncavallo : Pagliacci : Act 1 "Recitar!" [Canio] - Ruggero Leoncavallo - Movie
- La bohème: